# Tissue Classifier — Phase 1
Multiclass logistic regression on 1000 selected CpG markers.  
Stratified k-fold cross-validation with class weighting to handle imbalanced tissue representation.

## Imports

In [1]:
import sys
import warnings
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
    make_scorer,
)
from sklearn.preprocessing import LabelEncoder

# Add src/ to path so notebook can import pipeline modules
sys.path.insert(0, str(Path('../src')))
import data_loading
import config

KeyboardInterrupt: 

## Configuration

In [ ]:
# Tissue merge groups — must stay in sync with data_loading.build_merged_df()
# Keys are the merged label used in heatmap_matrix; values are the original raw labels.
MERGE_GROUPS = {
    'Eye_Retina':               ['Eye', 'Retina'],
    'Brain_Cortex_Subcortical': ['Brain_Cortex', 'Subcortical_Brain'],
    'Blood_Spleen_Thymus':      ['Blood', 'Spleen', 'Thymus'],
    'Skin_Ears_Tail':           ['Skin', 'Ears', 'Tail'],
}

# Tissues excluded from marker selection — samples are dropped from training
EXCLUDED_TISSUES = {'Sciatic_Nerve', 'Optic_Nerve', 'Mammary_Glands'}

# Build flat original_label -> merged_label lookup from MERGE_GROUPS
LABEL_MAP = {}
for merged_label, original_labels in MERGE_GROUPS.items():
    for orig in original_labels:
        LABEL_MAP[orig] = merged_label
# Tissues not in any merge group map to themselves

N_FOLDS     = 5
RANDOM_SEED = 42
FIGURES_DIR = config.FIGURES_DIR / 'classifier'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load data

In [ ]:
# Load selected CpG regions (tissues x probes, index = tissue names)
# Note: index_col=0 restores tissue names as row index
heatmap_matrix_path = config.DATA_DIR / 'processed' / 'top_50_regions_df.csv'
heatmap_matrix = pd.read_csv(heatmap_matrix_path, index_col=0)

print('heatmap_matrix shape (tissues x probes):', heatmap_matrix.shape)
heatmap_matrix

In [ ]:
# Load raw replicate beta matrix (probes x samples)
df_raw, _, _, _ = data_loading.load_all()

print('df_raw shape (probes x samples):', df_raw.shape)

## 2. Extract sample labels from column names

In [ ]:
# Column names follow the pattern <TissueType>.<replicate_n> (pandas suffix on read).
# Strip the suffix to recover the original tissue label per sample.
sample_cols = df_raw.columns[df_raw.columns != 'probe_ID'].tolist()

original_labels = pd.Series(
    [col.split('.')[0] for col in sample_cols],
    index=sample_cols,
    name='original_tissue'
)

# Sanity check: confirm all expected tissue types are present
print('Unique original tissue labels:', sorted(original_labels.unique()))

In [ ]:
# Map original labels to merged labels using LABEL_MAP.
# Tissues not in LABEL_MAP (unmerged ones) map to themselves.
merged_labels = original_labels.map(lambda x: LABEL_MAP.get(x, x))
merged_labels.name = 'merged_tissue'

print('Unique merged tissue labels:', sorted(merged_labels.unique()))

In [ ]:
# Exclude samples whose original label is in EXCLUDED_TISSUES
keep_mask       = ~original_labels.isin(EXCLUDED_TISSUES)
sample_cols_use = original_labels[keep_mask].index.tolist()
y_labels        = merged_labels[keep_mask]

n_excluded = (~keep_mask).sum()
print(f'Excluded {n_excluded} samples from: {EXCLUDED_TISSUES}')
print(f'Remaining samples: {len(sample_cols_use)}')
print()
print('Sample counts per merged tissue class:')
print(y_labels.value_counts().sort_index())

## 3. Build feature matrix X

In [ ]:
# Set probe_ID as index so we can mask rows by probe ID cleanly
df_raw_indexed = df_raw.set_index('probe_ID')

# Selected probe IDs from heatmap_matrix (columns of heatmap_matrix = probe IDs)
selected_probes = heatmap_matrix.columns.tolist()

# Sanity check: confirm all selected probes exist in df_raw
missing = set(selected_probes) - set(df_raw_indexed.index)
if missing:
    warnings.warn(f'{len(missing)} selected probes not found in df_raw: {missing}')
else:
    print(f'All {len(selected_probes)} selected probes found in df_raw.')

# Mask df_raw to selected probes, then select only the samples we're keeping,
# then transpose: result is samples x probes
X = (
    df_raw_indexed
    .loc[selected_probes, sample_cols_use]
    .T
)

print(f'X shape (samples x probes): {X.shape}')   # expected: (~268 x 1000)
assert X.shape[0] == len(y_labels), 'Sample count mismatch between X and y!'
assert X.shape[1] == len(selected_probes), 'Probe count mismatch!'

In [ ]:
# Check for NaN values — beta values should be complete after annotation filtering
n_nan = X.isna().sum().sum()
print(f'NaN values in X: {n_nan}')
if n_nan > 0:
    warnings.warn('NaN values found in feature matrix. Consider imputation before modelling.')

In [ ]:
# Encode string tissue labels to integers for sklearn
le = LabelEncoder()
y  = le.fit_transform(y_labels)

print('Class encoding:')
for i, cls in enumerate(le.classes_):
    print(f'  {i:2d}  {cls}')

## 4. Cross-validation setup

In [ ]:
# Warn if any class has fewer samples than N_FOLDS — stratification will fail or be unreliable
class_counts = y_labels.value_counts()
small_classes = class_counts[class_counts < N_FOLDS]
if not small_classes.empty:
    warnings.warn(
        f'The following classes have fewer than {N_FOLDS} samples and may cause '
        f'issues with stratified CV:\n{small_classes.to_string()}'
    )
else:
    print(f'All classes have >= {N_FOLDS} samples. Stratified {N_FOLDS}-fold CV is safe.')

In [ ]:
cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)

# Verify fold class distributions as a sanity check
print(f'Fold class distribution check ({N_FOLDS} folds):')
for fold_idx, (train_idx, val_idx) in enumerate(cv.split(X, y)):
    train_counts = pd.Series(y[train_idx]).value_counts().sort_index()
    val_counts   = pd.Series(y[val_idx]).value_counts().sort_index()
    print(f'  Fold {fold_idx + 1}: train={len(train_idx)} samples, val={len(val_idx)} samples, '
          f'min class in val={val_counts.min()}')

## 5. Train logistic regression with cross-validation

In [ ]:
model = LogisticRegression(
    multi_class='multinomial',
    solver='lbfgs',
    max_iter=1000,
    class_weight='balanced',   # compensates for imbalanced class sizes
    random_state=RANDOM_SEED,
    C=1.0,                     # inverse regularisation strength; tune later
)

scoring = {
    'balanced_accuracy': make_scorer(balanced_accuracy_score),
    'macro_f1':          make_scorer(f1_score, average='macro', zero_division=0),
}

cv_results = cross_validate(
    model,
    X, y,
    cv=cv,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1,
)

print('Cross-validation results:')
print(f"  Balanced accuracy : {cv_results['test_balanced_accuracy'].mean():.3f} "
      f"(+/- {cv_results['test_balanced_accuracy'].std():.3f})")
print(f"  Macro F1          : {cv_results['test_macro_f1'].mean():.3f} "
      f"(+/- {cv_results['test_macro_f1'].std():.3f})")
print()
print(f"  Train balanced accuracy: {cv_results['train_balanced_accuracy'].mean():.3f} "
      f"(+/- {cv_results['train_balanced_accuracy'].std():.3f})")
print(f"  Train macro F1         : {cv_results['train_macro_f1'].mean():.3f} "
      f"(+/- {cv_results['train_macro_f1'].std():.3f})")

## 6. Per-class metrics and confusion matrix
Re-fit manually fold-by-fold to accumulate predictions for a full confusion matrix.

In [ ]:
X_arr = X.values  # numpy array for indexing in loop

all_y_true = []
all_y_pred = []

for train_idx, val_idx in cv.split(X_arr, y):
    X_train, X_val = X_arr[train_idx], X_arr[val_idx]
    y_train, y_val = y[train_idx],     y[val_idx]

    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)

    all_y_true.extend(y_val)
    all_y_pred.extend(y_pred)

all_y_true = np.array(all_y_true)
all_y_pred = np.array(all_y_pred)

print('Classification report (aggregated across all folds):')
print(classification_report(
    all_y_true,
    all_y_pred,
    target_names=le.classes_,
    zero_division=0
))

In [ ]:
# Confusion matrix — normalised by true class (row) to account for imbalance
cm = confusion_matrix(all_y_true, all_y_pred, normalize='true')
cm_df = pd.DataFrame(cm, index=le.classes_, columns=le.classes_)

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(
    cm_df,
    cmap='Blues',
    vmin=0, vmax=1,
    annot=True,
    fmt='.2f',
    linewidths=0.4,
    ax=ax,
)
ax.set_xlabel('Predicted label')
ax.set_ylabel('True label')
ax.set_title('Normalised confusion matrix (stratified 5-fold CV)')
plt.tight_layout()

cm_path = FIGURES_DIR / f'confusion_matrix.{config.FIGURE_FORMAT}'
plt.savefig(cm_path, dpi=config.FIGURE_DPI, bbox_inches='tight')
plt.close('all')
print(f'Confusion matrix saved -> {cm_path}')

In [ ]:
# Per-class recall bar chart — highlights which tissue types are hardest to classify
per_class_recall = cm.diagonal()
recall_df = pd.DataFrame({
    'tissue':  le.classes_,
    'recall':  per_class_recall,
    'n_samples': [class_counts.get(cls, 0) for cls in le.classes_],
}).sort_values('recall')

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(recall_df['tissue'], recall_df['recall'], color='steelblue')
ax.set_xlabel('Recall (sensitivity)')
ax.set_title('Per-class recall — logistic regression (5-fold CV)')
ax.set_xlim(0, 1)
ax.axvline(x=recall_df['recall'].mean(), color='red', linestyle='--', label='Mean recall')
ax.legend()

# Annotate bars with sample count
for bar, n in zip(bars, recall_df['n_samples']):
    ax.text(
        bar.get_width() + 0.01,
        bar.get_y() + bar.get_height() / 2,
        f'n={n}',
        va='center', fontsize=8
    )

plt.tight_layout()

recall_path = FIGURES_DIR / f'per_class_recall.{config.FIGURE_FORMAT}'
plt.savefig(recall_path, dpi=config.FIGURE_DPI, bbox_inches='tight')
plt.close('all')
print(f'Per-class recall plot saved -> {recall_path}')

## 7. Summary

In [ ]:
print('===== SUMMARY =====')
print(f'Probes used:          {X.shape[1]}')
print(f'Samples used:         {X.shape[0]}')
print(f'Classes:              {len(le.classes_)}')
print(f'CV folds:             {N_FOLDS}')
print()
print(f"Val balanced accuracy : {cv_results['test_balanced_accuracy'].mean():.3f} "
      f"(+/- {cv_results['test_balanced_accuracy'].std():.3f})")
print(f"Val macro F1          : {cv_results['test_macro_f1'].mean():.3f} "
      f"(+/- {cv_results['test_macro_f1'].std():.3f})")
print()
print('Figures written to:', FIGURES_DIR)